## 1. Setup

In [1]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 7.7 MB/s eta 0:00:00ta 0:00:01


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient
from qdrant_client import QdrantClient

SEED = 42

COLLECTION_NAME = "products"

user_secrets = UserSecretsClient()

QDRANT_URL = user_secrets.get_secret("QDRANT_URL")
QDRANT_API_KEY = user_secrets.get_secret("QDRANT_API_KEY")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

print("Qdrant connection established.")
print("Collection:", COLLECTION_NAME)

Qdrant connection established.
Collection: products


In [5]:
print(client.get_collections())

collections=[CollectionDescription(name='products')]


## 2. Load Embedding Artifacts

In [6]:
PRODUCTS_PATH = Path(
    "/kaggle/input/notebooks/jndhruv/02-data-cleaning/"
    "multi-modal-fashion-ecom/data/processed/products.parquet"
)

products_df = pd.read_parquet(PRODUCTS_PATH)

print("Products:", len(products_df))

Products: 44419


In [7]:
IMAGE_EMBEDDINGS_PATH = Path("/kaggle/input/notebooks/jndhruv/03a-image-embeddings/image_embeddings.npy")
EMBEDDING_IDS_PATH = Path("/kaggle/input/notebooks/jndhruv/03a-image-embeddings/image_embedding_ids.npy")

image_embeddings = np.load(IMAGE_EMBEDDINGS_PATH)
embedding_ids = np.load(EMBEDDING_IDS_PATH)

print("Image embeddings:", image_embeddings.shape)
print("Embedding IDs:", embedding_ids.shape)
print("Embedding dtype:", image_embeddings.dtype)

Image embeddings: (44419, 512)
Embedding IDs: (44419,)
Embedding dtype: float32


In [9]:
assert image_embeddings.shape == (44_419, 512)
assert len(embedding_ids) == 44_419
assert len(np.unique(embedding_ids)) == 44_419

print("Image embedding artifacts validated.")

Image embedding artifacts validated.


In [10]:
SPARSE_VECTORS_PATH = Path("/kaggle/input/notebooks/jndhruv/04-sparse-embeddings/sparse_vectors.json")

with open(SPARSE_VECTORS_PATH) as f:
    sparse_vectors = json.load(f)

# JSON object keys are strings; Qdrant loader expects integer IDs.
sparse_vectors = {
    int(pid): vector
    for pid, vector in sparse_vectors.items()
}

print("Sparse vectors:", len(sparse_vectors))

Sparse vectors: 44419


## 3. Mapping

In [12]:
dense_vectors = dict(
    zip(
        embedding_ids.tolist(),
        image_embeddings.tolist()
    )
)

print("Dense vectors:", len(dense_vectors))

Dense vectors: 44419


In [13]:
product_ids = set(products_df["id"].astype(int))
dense_ids = set(dense_vectors.keys())
sparse_ids = set(sparse_vectors.keys())

assert product_ids == dense_ids
assert product_ids == sparse_ids

print("All three artifacts contain the exact same product IDs.")
print("Products ready for ingestion:", len(product_ids))

All three artifacts contain the exact same product IDs.
Products ready for ingestion: 44419


## 4. Load Qdrant Loader Scripts

In [16]:
!git clone https://github.com/jndhruv/multi-modal-fashion-ecom.git
!ls multi-modal-fashion-ecom/data/scripts

fatal: destination path 'multi-modal-fashion-ecom' already exists and is not an empty directory.
clean_data.py	      generate_sparse_vectors.py
create_collection.py  load_to_qdrant.py


In [17]:
import sys

sys.path.append(
    "/kaggle/working/multi-modal-fashion-ecom/data/scripts"
)

from load_to_qdrant import upsert_products
from create_collection import COLLECTION_NAME

print("Loader imported successfully.")
print("Collection:", COLLECTION_NAME)

Loader imported successfully.
Collection: products


## 5. Uploading to Qdrant Collection

In [18]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Status:", collection_info.status)
print("Points:", collection_info.points_count)

assert str(collection_info.status).lower().endswith("green")
assert collection_info.points_count == 0

Collection: products
Status: green
Points: 0


In [19]:
upsert_products(
    client=client,
    df=products_df,
    dense_vectors=dense_vectors,
    sparse_vectors=sparse_vectors,
    batch_size=256,
)

Upserted 256/44419
Upserted 512/44419
Upserted 768/44419
Upserted 1024/44419
Upserted 1280/44419
Upserted 1536/44419
Upserted 1792/44419
Upserted 2048/44419
Upserted 2304/44419
Upserted 2560/44419
Upserted 2816/44419
Upserted 3072/44419
Upserted 3328/44419
Upserted 3584/44419
Upserted 3840/44419
Upserted 4096/44419
Upserted 4352/44419
Upserted 4608/44419
Upserted 4864/44419
Upserted 5120/44419
Upserted 5376/44419
Upserted 5632/44419
Upserted 5888/44419
Upserted 6144/44419
Upserted 6400/44419
Upserted 6656/44419
Upserted 6912/44419
Upserted 7168/44419
Upserted 7424/44419
Upserted 7680/44419
Upserted 7936/44419
Upserted 8192/44419
Upserted 8448/44419
Upserted 8704/44419
Upserted 8960/44419
Upserted 9216/44419
Upserted 9472/44419
Upserted 9728/44419
Upserted 9984/44419
Upserted 10240/44419
Upserted 10496/44419
Upserted 10752/44419
Upserted 11008/44419
Upserted 11264/44419
Upserted 11520/44419
Upserted 11776/44419
Upserted 12032/44419
Upserted 12288/44419
Upserted 12544/44419
Upserted 1280

## 6. Verifying Qdrant Population

In [20]:
exact_count = client.count(
    collection_name=COLLECTION_NAME,
    exact=True,
)

print("Exact point count:", exact_count.count)

assert exact_count.count == 44_419

print("Qdrant ingestion verified.")

Exact point count: 44419
Qdrant ingestion verified.
